In [7]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt
import os

In [8]:
!pip install matplotlib>=3.7.0

In [9]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

Using device: cuda


In [10]:
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.Grayscale(num_output_channels=3),  # X光通常是灰度 → 转3通道
    transforms.ToTensor(),
    transforms.Normalize([0.5], [0.5])
])

In [11]:
train_dataset = datasets.ImageFolder("newdata/train", transform=transform)
test_dataset = datasets.ImageFolder("newdata/test", transform=transform)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

print("Classes:", train_dataset.classes)  # ['covid', 'non']

Classes: ['covid', 'non']


In [12]:
model = models.resnet18(pretrained=True)

# 修改最后一层 → 二分类
model.fc = nn.Linear(model.fc.in_features, 2)

model = model.to(device)

c:\Users\ajifang\anaconda3\envs\torch220cu121\lib\site-packages\torchvision\models\_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
c:\Users\ajifang\anaconda3\envs\torch220cu121\lib\site-packages\torchvision\models\_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet18_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet18_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)
Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to C:\Users\ajifang/.cache\torch\hub\checkpoints\resnet18-f37072fd.pth
100.0%


In [13]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.0001)

In [14]:
def train(model, loader, optimizer, criterion):
    model.train()
    total_loss = 0
    correct = 0
    
    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)
        
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item()
        _, preds = torch.max(outputs, 1)
        correct += (preds == labels).sum().item()
    
    acc = correct / len(loader.dataset)
    return total_loss / len(loader), acc

In [15]:
def evaluate(model, loader):
    model.eval()
    correct = 0
    
    with torch.no_grad():
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            _, preds = torch.max(outputs, 1)
            correct += (preds == labels).sum().item()
    
    acc = correct / len(loader.dataset)
    return acc

In [16]:
epochs = 10

for epoch in range(epochs):
    train_loss, train_acc = train(model, train_loader, optimizer, criterion)
    test_acc = evaluate(model, test_loader)
    
    print(f"Epoch {epoch+1}/{epochs}")
    print(f"Train Loss: {train_loss:.4f}, Train Acc: {train_acc:.4f}")
    print(f"Test Acc: {test_acc:.4f}")
    print("-"*30)

Epoch 1/10
Train Loss: 0.1785, Train Acc: 0.9170
Test Acc: 0.9967
------------------------------
Epoch 2/10
Train Loss: 0.0240, Train Acc: 0.9986
Test Acc: 0.9967
------------------------------
Epoch 3/10
Train Loss: 0.0128, Train Acc: 0.9981
Test Acc: 0.9956
------------------------------
Epoch 4/10
Train Loss: 0.0030, Train Acc: 1.0000
Test Acc: 0.9972
------------------------------
Epoch 5/10
Train Loss: 0.0020, Train Acc: 1.0000
Test Acc: 0.9972
------------------------------
Epoch 6/10
Train Loss: 0.0016, Train Acc: 1.0000
Test Acc: 0.9972
------------------------------
Epoch 7/10
Train Loss: 0.0012, Train Acc: 1.0000
Test Acc: 0.9978
------------------------------
Epoch 8/10
Train Loss: 0.0011, Train Acc: 1.0000
Test Acc: 0.9978
------------------------------
Epoch 9/10
Train Loss: 0.0008, Train Acc: 1.0000
Test Acc: 0.9956
------------------------------
Epoch 10/10
Train Loss: 0.0009, Train Acc: 1.0000
Test Acc: 0.9972
------------------------------


In [17]:
torch.save(model.state_dict(), "covid_classifier.pth")